# UrbanGap: source data and approved study area

Explore the public-toilet source and understand how the established-urban prediction area is preserved. This notebook reads pipeline outputs; it does not overwrite datasets. Run `python scripts/run_pipeline.py` first.

Inputs: immutable toilet CSV; interim preliminary/final SA1 GeoJSON; study selection registry. Outputs: summary tables and a map. See `docs/methodology.md` for the distinction between the approved geography and the 500 people/km² urban-core threshold.

In [ ]:
from pathlib import Path
import sys

# Works when the kernel starts in the project root or notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from urbangap.common import INTERIM, PROCESSED, CRS

assert (PROCESSED / "sydney_hex_features.gpkg").exists(), "Run python scripts/run_pipeline.py first."


## The placement proxy

A mapped facility is a historical infrastructure record, not evidence that provision is optimal. Facility counts do not measure cubicles, capacity or opening hours. The grid uses national records directly, avoiding the historical bounding-box CSV branch.

In [ ]:
toilets = pd.read_csv(ROOT / "data/raw/national_toilets.csv")
assert toilets.FacilityID.is_unique
display(toilets.FacilityType.value_counts().head(12).rename("records").to_frame())
print(f"National records: {len(toilets):,}")

## Fixed geographic decisions

The saved preliminary area reproduces at density 50, despite the former saved count/title indicating 500. Positional cluster IDs are unstable. Production now uses explicit approved preliminary SA1 membership, recalculates the named growth-suburb exclusions, and asserts the exact approved final membership. The active urban-core threshold is 500; it does not silently redefine the approved boundary. The registry is a versioned geographic decision input, not a feature cache.

In [ ]:
preliminary = gpd.read_file(INTERIM / "boundaries/sydney_study_sa1_preliminary.geojson")
study = gpd.read_file(INTERIM / "boundaries/sydney_study_sa1_final.geojson")
from urbangap.study import DENSITY_THRESHOLD
from urbangap.common import read_ids
assert len(preliminary) == 10753
assert len(study) == 10503
assert set(study.SA1_CODE21) == read_ids(ROOT / "docs/approved_final_sa1_ids.txt")
print("Urban-core threshold:", DENSITY_THRESHOLD, "people/km²")
print("Final study population:", int(study.population.sum()))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
preliminary.plot(ax=ax, color="lightgrey", linewidth=0)
study.plot(ax=ax, color="steelblue", linewidth=0)
ax.set(title="Approved study area; excluded growth areas in grey")
ax.set_axis_off()
plt.show()